In [ ]:
!pip install amazon-transcribe sounddevice

In [4]:
import asyncio
import numpy as np
import sounddevice as sd

from amazon_transcribe.client import TranscribeStreamingClient
from amazon_transcribe.handlers import TranscriptResultStreamHandler
from amazon_transcribe.model import TranscriptEvent

In [5]:
REGION = "us-east-1"       # change if you want, must match where you use Transcribe
LANGUAGE_CODE = "en-US"
SAMPLE_RATE = 48000
CHUNK_MS = 20
FRAMES_PER_CHUNK = int(SAMPLE_RATE * CHUNK_MS / 1000)

VOCAB_NAME = None  # e.g. "VocabularyTable" (name only, NOT S3 path)

In [ ]:
import asyncio
import numpy as np
import sounddevice as sd

from amazon_transcribe.client import TranscribeStreamingClient
from amazon_transcribe.handlers import TranscriptResultStreamHandler
from amazon_transcribe.model import TranscriptEvent

from IPython.display import clear_output

REGION = "us-east-1"
LANGUAGE_CODE = "en-US"

INPUT_DEVICE_INDEX = 17
SAMPLE_RATE = 48000
CHUNK_MS = 20
FRAMES_PER_CHUNK = int(SAMPLE_RATE * CHUNK_MS / 1000)

VOCAB_NAME = None  # e.g. "VocabularyTable"
chunks = 0
level = {"n": 0}


from IPython.display import clear_output

class DebugTranscriptHandler(TranscriptResultStreamHandler):
    async def handle_transcript_event(self, transcript_event: TranscriptEvent):
        # If this prints, Transcribe is sending transcript events
        print("Transcript event received", flush=True)

        for result in transcript_event.transcript.results:
            if not result.alternatives:
                continue
            text = result.alternatives[0].transcript.strip()
            if not text:
                continue

            if result.is_partial:
                clear_output(wait=True)
                print("PARTIAL:", text, flush=True)
            else:
                print("FINAL:", text, flush=True)


async def run_realtime_transcribe_live():
    sd.default.device = (INPUT_DEVICE_INDEX, None)

    client = TranscribeStreamingClient(region=REGION)
    stream = await client.start_stream_transcription(
        language_code=LANGUAGE_CODE,
        media_sample_rate_hz=SAMPLE_RATE,
        media_encoding="pcm",
        vocabulary_name=VOCAB_NAME,
    )

    loop = asyncio.get_running_loop()
    q: asyncio.Queue[bytes] = asyncio.Queue()

    def callback(indata, frames, time_info, status):
        try:
            # compute loudness
            rms = float(np.sqrt(np.mean(indata**2)))
            level["n"] += 1
            if level["n"] % 50 == 0:  # ~ once per second
                print(f"RMS level: {rms:.6f}", flush=True)

            pcm16 = (np.clip(indata, -1, 1) * 32767).astype(np.int16).tobytes()
            loop.call_soon_threadsafe(q.put_nowait, pcm16)

        except Exception as e:
            print("Callback error:", e, flush=True)


    async def write_audio():
        with sd.InputStream(
            samplerate=SAMPLE_RATE,
            channels=1,
            dtype="float32",
            blocksize=FRAMES_PER_CHUNK,
            callback=callback,
        ):
            while True:
                chunk = await q.get()
                await stream.input_stream.send_audio_event(audio_chunk=chunk)

    handler = LiveTranscriptHandler(stream.output_stream)
    await asyncio.gather(write_audio(), handler.handle_events())


# In Jupyter:
await run_realtime_transcribe_live()


Audio chunks sent: 50
Audio chunks sent: 100
Audio chunks sent: 150
Audio chunks sent: 200
Audio chunks sent: 250
Audio chunks sent: 300
Audio chunks sent: 350
Audio chunks sent: 400
Audio chunks sent: 450
Audio chunks sent: 500
Audio chunks sent: 550
Audio chunks sent: 600
Audio chunks sent: 650
Audio chunks sent: 700
Audio chunks sent: 750
Audio chunks sent: 800
Audio chunks sent: 850
Audio chunks sent: 900
Audio chunks sent: 950
Audio chunks sent: 1000
Audio chunks sent: 1050
Audio chunks sent: 1100
Audio chunks sent: 1150
Audio chunks sent: 1200
Audio chunks sent: 1250
Audio chunks sent: 1300
Audio chunks sent: 1350
Audio chunks sent: 1400
Audio chunks sent: 1450
Audio chunks sent: 1500
Audio chunks sent: 1550
Audio chunks sent: 1600
Audio chunks sent: 1650
Audio chunks sent: 1700
Audio chunks sent: 1750


CancelledError: 

In [4]:
import sounddevice as sd
sd.query_devices()

   0 Microsoft Sound Mapper - Input, MME (2 in, 0 out)
>  1 Microphone Array (Intel® Smart , MME (4 in, 0 out)
   2 Jack Mic (Realtek(R) Audio), MME (2 in, 0 out)
   3 Microsoft Sound Mapper - Output, MME (0 in, 2 out)
<  4 Headphones (Realtek(R) Audio), MME (0 in, 8 out)
   5 Speakers (Realtek(R) Audio), MME (0 in, 8 out)
   6 27N1A (NVIDIA High Definition A, MME (0 in, 2 out)
   7 Primary Sound Capture Driver, Windows DirectSound (2 in, 0 out)
   8 Microphone Array (Intel® Smart Sound Technology for Digital Microphones), Windows DirectSound (4 in, 0 out)
   9 Jack Mic (Realtek(R) Audio), Windows DirectSound (2 in, 0 out)
  10 Primary Sound Driver, Windows DirectSound (0 in, 8 out)
  11 Headphones (Realtek(R) Audio), Windows DirectSound (0 in, 8 out)
  12 Speakers (Realtek(R) Audio), Windows DirectSound (0 in, 8 out)
  13 27N1A (NVIDIA High Definition Audio), Windows DirectSound (0 in, 2 out)
  14 Headphones (Realtek(R) Audio), Windows WASAPI (0 in, 8 out)
  15 Speakers (Realtek(R) Au

In [5]:
import sounddevice as sd
sd.default.device = (17, None) 

In [7]:
import numpy as np
import sounddevice as sd

sd.default.device = (17, None)

SAMPLE_RATE = 48000  # <-- change from 16000 to 48000
audio = sd.rec(int(2 * SAMPLE_RATE), samplerate=SAMPLE_RATE, channels=1, dtype="float32")
sd.wait()

print("Max amplitude:", float(np.max(np.abs(audio))))

Max amplitude: 6.535711971622504e-09
